In [0]:
%run ../../config/utils

In [0]:
"""Spark job to create an ingestable coupon list

TODO: Add integration test
"""

from datetime import datetime, timedelta

import pyspark.sql.functions as sqlf
import pyspark.sql.types as sqlt
from pyspark import StorageLevel

import sys
sys.path.append("..")
sys.path.append("../..")

from lib_assignment.assn_io import JobManager
from lib_assignment.checks import (
    check_execution_overwrite,
    check_prod_status,
)
from lib_assignment.coupon_utils import (
    add_coupons,
    append_new_campaign,
    append_non_duplicates,
    calculate_discount,
    calculate_member_trips,
    calculate_member_usage,
    clean_cpg_coupon_file,
    format_coupons,
    remove_exclusions,
    safe_cast_by_schema
)
from lib_assignment.schemas.coupon_schemas import (
    BASKET_COUPON_SCHEMA,
    CAT_COUPON_SCHEMA,
    CPN_BNK_SCHEMA,
    CPN_DISCOUNT_SCHEMA,
    CPN_MAP_SCHEMA,
    CPN_QUALS_SCHEMA,
    MEM_TRIP_SCHEMA,
    MEM_USAGE_SCHEMA,
)
from lib.iotools import write_local_to_s3
from lib_assignment.assn_utils import env_path


In [0]:
env = dbutils.widgets.get("environment").lower()
spark.conf.set("spark.sql.ansi.enabled", "false")

In [0]:
def has_coupons(job):
    """Identify whether the campaign has coupons"""
    if (
        job.config.paths.get("CPG_COUPON_LIST_PATH")
        or job.config.paths.get("CATEGORY_COUPON_PATH")
        or job.config.paths.get("BASKET_COUPON_PATH")
    ):
        return True
    return False


def initialize_dfs(job):
    """Initialize the dfs required to store the final outputted data"""
    for name, schema in (
        ("quals", CPN_QUALS_SCHEMA),
        ("memtrips", MEM_TRIP_SCHEMA),
        ("bank", CPN_BNK_SCHEMA),
        ("coupmap", CPN_MAP_SCHEMA),
        ("memusage", MEM_USAGE_SCHEMA),
        ("discounts", CPN_DISCOUNT_SCHEMA),
    ):
        tbl = job.spark.createDataFrame([], schema)
        job.data.tables[name] = tbl


def prepare_data(job):
    """Prepare input data"""
    job.data.read("article_dna", fs_article_nbr_category_dna_full, filetype="table")
    job.data.tables["article_dna"] = job.data.tables[
        "article_dna"
    ].dropDuplicates(["ARTICLE_NBR"]).withColumn('article_nbr',sqlf.col('ARTICLE_NBR').try_cast('long'))
    job.data.read("article_map", silver_master_item, filetype="table")
    job.data.tables["article_map"] = job.data.tables[
        "article_map"
    ].dropDuplicates(["ARTICLE_NBR"]).withColumn('article_nbr',sqlf.col('ARTICLE_NBR').try_cast('long'))
    job.data.read("ah5_dna", fs_ah5_cd_category_dna_full, filetype="table")
    job.data.tables["ah5_dna"] = job.data.tables["ah5_dna"].dropDuplicates(
        ["AH5_DESC"]
    )
    job.data.read("ah4_dna", fs_ah4_cd_category_dna_full, filetype="table")
    job.data.tables["ah4_dna"] = job.data.tables["ah4_dna"].dropDuplicates(
        ["AH4_DESC"]
    )
    job.data.read("transactions", silver_transaction_fiscal_detail, filetype="table")
    job.data.tables["transactions"] = job.data.tables["transactions"].withColumn('ARTICLE_NBR', sqlf.col('ARTICLE_NBR').try_cast('long'))
    job.data.read("item", silver_master_item, filetype="table")
    job.data.tables["item"] = job.data.tables["item"].withColumn('ARTICLE_NBR', sqlf.col('ARTICLE_NBR').try_cast('long'))

    # prep transactions
    trans_end = datetime.strptime(
        job.config.params["assignment_date"], "%Y-%m-%d"
    )
    trans_start = trans_end - timedelta(days=(365))
    member_trips = calculate_member_trips(
        job.data.tables["transactions"],
        job.data.tables["item"],
        trans_start,
        trans_end,
    )
    # member_trips.persist(StorageLevel.DISK_ONLY)

    # prep category_dna
    article_map = job.data.tables["article_map"].withColumn('article_nbr',sqlf.col('ARTICLE_NBR').try_cast('long'))
    article_ah5_ah4 = article_map.select(
        "article_nbr", "AH5_CD", "AH5_DESC", "AH4_CD", "AH4_DESC"
    )
    article_map = job.data.tables["article_map"].select(
        "article_nbr", "MCH3_DESC", "AH5_CD", "AH5_DESC"
    ).withColumn('article_nbr',sqlf.col('ARTICLE_NBR').try_cast('long'))
    article_map = article_map.filter(
        article_map.MCH3_DESC.isin(
            job.config.params["backfill"]["substitutable_MCH3_exclusion"]
        )
    )
    article_map = article_map.select(
        "article_nbr", "AH5_CD", "AH5_DESC"
    ).distinct()
    article_dna = job.data.tables["article_dna"].drop(
        "AH5_CD", "AH4_CD", "AH5_DESC", "AH4_DESC"
    ).withColumn('article_nbr',sqlf.col('ARTICLE_NBR').try_cast('long'))
    article_map = article_map.join(article_dna, ["article_nbr"], "left")
    article_map = article_map.filter(article_map.AH5_CD.isNotNull())
    article_dna = job.data.tables["article_dna"]
    article_dna = article_dna.withColumnRenamed("article_nbr", "category")
    article_dna = article_dna.withColumnRenamed(
        "ARTICLE_DESC", "category_desc"
    )
    ah5_dna = job.data.tables["ah5_dna"]
    ah5_dna = ah5_dna.withColumnRenamed("AH5_CD", "category")
    ah5_dna = ah5_dna.withColumnRenamed("AH5_DESC", "category_desc")
    ah4_dna = job.data.tables["ah4_dna"]
    ah4_dna = ah4_dna.withColumnRenamed("AH4_CD", "category")
    ah4_dna = ah4_dna.withColumnRenamed("AH4_DESC", "category_desc")
    article_dna = article_dna.drop("AH5_CD", "AH5_DESC", "AH4_CD", "AH4_DESC")
    cat_dna = article_dna.union(ah5_dna)
    cat_dna = cat_dna.union(ah4_dna)
    cat_dna = cat_dna.withColumn("category", cat_dna.category.try_cast("integer"))
    # cat_dna.persist(StorageLevel.DISK_ONLY)

    for name, table in (
        ("member_trips", member_trips),
        ("article_map", article_map),
        ("ah5_dna", ah5_dna),
        ("ah4_dna", ah4_dna),
        ("cat_dna", cat_dna),
        ("article_ah5_ah4", article_ah5_ah4),
    ):
        job.data.tables[name] = table


def create_coupons(job):

    quals = job.data.tables["quals"]
    bank = job.data.tables["bank"]
    coupmap = job.data.tables["coupmap"]
    memtrips = job.data.tables["memtrips"]
    cat_dna = job.data.tables["cat_dna"]
    member_trips = job.data.tables["member_trips"]
    discounts = job.data.tables["discounts"]

    category = basket = article = prices = False
    # handle basket type offers
    if job.config.paths.get("BASKET_COUPON_PATH"):
        basket = True
        print("    Creating basket coupons...")

        # 1. Read in data
        basket_coups = job.spark.read.csv(
            env_path(job.config.paths["BASKET_COUPON_PATH"], output_vol, env),
            header=True,
            schema=BASKET_COUPON_SCHEMA,
        )
        basket_cnt = basket_coups.count()
        print("    num basket coupons: {}".format(basket_cnt))
        # 2. Format data
        basket, basket_map, _, _ = format_coupons(
            basket_coups, member_trips, cat_dna, job.config.params, "basket"
        )
        # 3. attach to dynamic tables
        quals = add_coupons(quals, basket, subset=quals.columns)
        bank = add_coupons(bank, basket, subset=bank.columns)
        coupmap = add_coupons(coupmap, basket_map, subset=coupmap.columns)

    # handle category type offers
    if job.config.paths.get("CATEGORY_COUPON_PATH"):
        category = True
        print("    Creating category coupons...")

        # 1. Read in data
        cat_coups = job.spark.read.csv(
            env_path(job.config.paths["CATEGORY_COUPON_PATH"], output_vol, env),
            header=True,
            inferSchema=True,
        )
        cat_coups = cat_coups.withColumn(
            "cpn_ah4_cd",
            sqlf.explode(
                sqlf.when(
                    sqlf.split(cat_coups.cpn_ah4_cd, ",").isNotNull(),
                    sqlf.split(cat_coups.cpn_ah4_cd, ","),
                ).otherwise(sqlf.array(sqlf.lit(None).try_cast(sqlt.LongType())))
            ),
        )
        cat_coups = cat_coups.withColumn(
            "cpn_ah4_cd", sqlf.trim(cat_coups.cpn_ah4_cd)
        )
        cat_coups = cat_coups.withColumn(
            "cpn_ah5_cd",
            sqlf.explode(
                sqlf.when(
                    sqlf.split(cat_coups.cpn_ah5_cd, ",").isNotNull(),
                    sqlf.split(cat_coups.cpn_ah5_cd, ","),
                ).otherwise(sqlf.array(sqlf.lit(None).try_cast(sqlt.LongType())))
            ),
        )
        cat_coups = cat_coups.withColumn(
            "cpn_ah5_cd", sqlf.trim(cat_coups.cpn_ah5_cd)
        )
        for column in CAT_COUPON_SCHEMA:
            cat_coups = cat_coups.withColumn(
                column.name, sqlf.col(column.name).try_cast(column.dataType)
            )

        cat_cnt = cat_coups.select("cpn_nbr").distinct().count()
        print("    num category coupons: {}".format(cat_cnt))

        # 2. Format data
        cat, cat_map, cat_memtrips, _ = format_coupons(
            cat_coups, member_trips, cat_dna, job.config.params, "category"
        )

        # 3. attach to dynamic tables
        memtrips = add_coupons(memtrips, cat_memtrips)
        quals = add_coupons(quals, cat, subset=quals.columns)
        bank = add_coupons(bank, cat, subset=bank.columns)
        coupmap = add_coupons(coupmap, cat_map, subset=coupmap.columns)

    # handle article type offers
    if job.config.paths.get("CPG_COUPON_LIST_PATH"):
        article = True
        print("    Creating article coupons...")

        # 1. Read in data and clean
        cpg_coups = job.spark.read.csv(
            env_path(job.config.paths["CPG_COUPON_LIST_PATH"], output_vol, env), header=True
        )
        cpg_coups = clean_cpg_coupon_file(
            cpg_coups, job.config.params["coupon_types"]
        )
        # removing exclusions, need to rename column to preserve general excl. fn.
        cpg_coups = cpg_coups.withColumnRenamed("article_nbr", "category")
        cpg_coups, exclusion_log = remove_exclusions(
            cpg_coups,
            cat_dna,
            job.data.tables["article_ah5_ah4"],
            job.config.params["exclusion_types"],
            job.config.params["excluded_coupons_force_back_in"],
        )
        cpg_coups = cpg_coups.withColumnRenamed("category", "article_nbr")
        cpg_cnt = cpg_coups.select("cpn_nbr").distinct().count()
        print("    num article coupons: {}".format(cpg_cnt))

        # 2. Format data
        cpg, cpg_map, cpg_memtrips, cpn_excl_log = format_coupons(
            cpg_coups,
            member_trips,
            cat_dna,
            job.config.params,
            "article",
            job.data.tables["article_map"],
        )
        # 3. attach to dynamic tables
        memtrips = add_coupons(memtrips, cpg_memtrips).dropna(how="all")
        quals = add_coupons(quals, cpg, subset=quals.columns).dropna(how="all")
        bank = add_coupons(bank, cpg, subset=bank.columns).dropna(how="all")
        coupmap = add_coupons(coupmap, cpg_map, subset=coupmap.columns).dropna(
            how="all"
        )

    if job.config.paths.get("COUPON_PRICE_PATH"):
        prices = True
        print("    Creating coupon discounts...")
        article_prices = job.spark.read.csv(
            env_path(job.config.paths["COUPON_PRICE_PATH"], output_vol, env), header=True
        )
        discounts = calculate_discount(article_prices)

    for name, table in (
        ("quals", quals),
        ("bank", bank),
        ("coupmap", coupmap),
    ):
        job.data.tables[name] = table

    # Member usage table, dependent on some above tables
    trans_end = datetime.strptime(
        job.config.params["assignment_date"], "%Y-%m-%d"
    )
    trans_start = trans_end - timedelta(days=(365))
    memusage = calculate_member_usage(
        job.data.tables["transactions"],
        job.data.tables["item"],
        quals,
        coupmap,
        trans_start,
        trans_end,
        job.config.params["experiment"],
    )
    job.data.tables["memusage"] = memusage

    if prices:
        job.data.tables["discounts"] = discounts
    if category or article:
        job.data.tables["memtrips"] = memtrips
        job.data.tables["memusage"] = memusage
    if article:
        return exclusion_log, cpn_excl_log

    return None, None


def write(job):
    """Write coupons out to final coupon bank"""

    quals = job.data.tables["quals"]
    bank = job.data.tables["bank"]
    coupmap = job.data.tables["coupmap"]
    memtrips = job.data.tables["memtrips"]
    memusage = job.data.tables["memusage"]
    discounts = job.data.tables["discounts"]

    PATHS = job.config.paths

    print("    total number of coupons: {}".format(bank.count()))

    discounts.write.csv(
        env_path(PATHS["COUPON_DISCOUNT"], output_vol, env), mode="overwrite", header=True
    )

    if bank.count() > 2:
        memtrips = memtrips.repartition(int(bank.count() / 2))
        memusage = memusage.repartition(int(bank.count() / 2))
    memtrips.write.parquet(env_path(PATHS["COUPON_MEMTRIP"], output_vol, env), mode="overwrite")
    print("finished printing COUPON_MEMTRIP")
    memusage.write.parquet(env_path(PATHS["COUPON_MEMUSAGE"], output_vol, env), mode="overwrite")
    print("finished printing COUPON_MEMUSAGE")

    if job.config.params["run_type"].lower() == "prod":
        if has_coupons(job):
            job.data.read("existing_quals", "COUPON_QUALS", filetype="csv")
            job.data.tables["existing_quals"] = safe_cast_by_schema(job.data.tables["existing_quals"], CPN_QUALS_SCHEMA)
            append_non_duplicates(
                job.spark,
                job.data.tables["existing_quals"],
                quals,
                PATHS["COUPON_QUALS"],
                "csv",
                "experiment_id",
                output_vol,
                env
            )

            job.data.read("updated_quals", "COUPON_QUALS", filetype="csv")
            updated_quals = job.data.tables["updated_quals"]

            append_new_campaign(
                job,
                coupmap,
                "COUPON_MAP",
                "csv",
                updated_quals,
                ["cpn_nbr", "article_nbr"],
                CPN_MAP_SCHEMA
            )
            append_new_campaign(
                job,
                bank,
                "COUPON_BANK",
                "csv",
                updated_quals,
                ["cpn_nbr", "cpn_desc"],
                CPN_BNK_SCHEMA
            )

        else:
            print(
                "No new coupons, not adding to production coupon bank"
            )
    else:
        for tbl, path in (
            (quals, "COUPON_QUALS"),
            (bank, "COUPON_BANK"),
            (coupmap, "COUPON_MAP"),
        ):
            if tbl.count() == 0:
                tbl = tbl.columns
                tbl = job.sc.parallelize([tbl]).toDF(["header"])
                # tbl = job.spark.createDataFrame(tbl, StringType())
                tbl.coalesce(1).write.csv(
                    env_path(PATHS[path], output_vol, env), mode="overwrite", header=False
                )
            else:
                tbl.coalesce(1).write.csv(
                    env_path(PATHS[path], output_vol, env), mode="overwrite", header=True
                )
            print("finished printing {}".format(path))


In [0]:
name = "CreateCoupons"
job = JobManager("coupon_creation", "../config/config_template.yml", spark, output_vol, env)

if job.config.params["run_type"].lower() == "prod":
    check_execution_overwrite(
        job,
        paths_to_check=[job.config.paths["COUPON_MEMTRIP"]]
    )
    check_prod_status(job)

print("1. Initialize coupon dataframes")
initialize_dfs(job)

if has_coupons(job):
    print("2. Read in reference data and prepare static data")
    prepare_data(job)

    print("3. Creating Coupons...")
    exclusion_log, cpn_excl_log = create_coupons(job)
else:
    print("Skipping steps 2 & 3 - no coupons provided")

print("4. Write output to file...")
write(job)

print("5. Write logging file and shut down...")

if job.config.paths.get("CPG_COUPON_LIST_PATH"):
    exclusion_log.coalesce(1).write.csv(
        env_path(job.config.paths["COUPON_EXCLUSION_LOG"], output_vol, env),
        mode="overwrite",
        header=True,
    )
    cpn_excl_log.coalesce(1).write.csv(
        env_path(job.config.paths["COUPON_BACKFILL_ELIGIBILITY_LOG"], output_vol, env),
        mode="overwrite",
        header=True,
    )

if job.config.params["run_type"].lower() == "prod":
    log_line = {
        "campaign": job.config.params["campaign"],
        "run_name": job.config.params["run_name"],
        "run_type": job.config.params["run_type"],
        "log_event": "END",
        "time": str(datetime.now().strftime("%Y-%m-%d-%H-%M-%S")),
    }

    write_local_to_s3(log_line, job.config.paths["COUPON_LOG"], 'overwrite', output_vol, env)

print("done")